# Module 1: Panel Data Exploration

## Learning Objectives

By completing this notebook, you will:
1. Visualize panel data effectively using time series and cross-sectional plots
2. Compute and interpret within and between summary statistics
3. Assess variation structure to guide model selection
4. Identify trends, seasonality, and entity heterogeneity
5. Detect outliers and data quality issues in panel data

## Prerequisites

- Module 1.1 (Data Preparation) completed
- Understanding of summary statistics
- Basic visualization skills

## Estimated Time

60-75 minutes

---

## Setup

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Configure plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Set random seed
np.random.seed(42)

print("✅ Setup complete")

In [ ]:
section_divider("1. Generate Example Panel Data")

In [ ]:
learning_objectives(["Visualize panel data effectively using time series and cross-sectional plots", "Compute and interpret within and between summary statistics", "Assess variation structure to guide model selection", "Identify trends, seasonality, and entity heterogeneity", "Detect outliers and data quality issues in panel data"])

In [ ]:
apply_course_theme()
apply_plot_theme()

## 1. Generate Example Panel Data

We'll create firm-level panel data with rich variation patterns.

In [ ]:
# Generate firm panel data
n_firms = 50
n_years = 15
years = np.arange(2008, 2008 + n_years)
firm_ids = np.arange(1, n_firms + 1)

# Firm characteristics (time-invariant)
firm_quality = np.random.randn(n_firms) * 0.5 + 1.0  # Centered at 1
firm_industry = np.random.choice(['Tech', 'Manufacturing', 'Services', 'Retail'], n_firms)

# Generate panel data
data = []
for i, firm_id in enumerate(firm_ids):
    # Firm-specific trend
    firm_trend = firm_quality[i] * 0.05
    
    for t, year in enumerate(years):
        # Sales (with trend, seasonality, and shocks)
        trend_component = 100 + firm_trend * t * 10
        seasonal_component = 20 * np.sin(2 * np.pi * t / 4)  # 4-year cycle
        firm_effect = firm_quality[i] * 50
        
        # Financial crisis shock (2008-2009)
        crisis_shock = -30 if year in [2008, 2009] else 0
        
        # COVID shock (2020)
        covid_shock = -40 if year == 2020 else 0
        
        sales = (trend_component + seasonal_component + firm_effect + 
                crisis_shock + covid_shock + np.random.randn() * 15)
        
        # R&D spending (persistent, correlated with firm quality)
        rd_base = firm_quality[i] * 20 + 10
        rd_spending = rd_base + np.random.randn() * 3
        
        # Employment (sticky, slow adjustment)
        emp_base = firm_quality[i] * 100 + 200
        employment = emp_base * (1.01 ** t) + np.random.randn() * 10
        
        # Profit margin (volatile)
        profit_margin = 0.15 + firm_quality[i] * 0.05 + np.random.randn() * 0.03
        profit_margin = max(0.01, min(0.40, profit_margin))  # Constrain
        
        data.append({
            'firm_id': firm_id,
            'year': year,
            'sales': sales,
            'rd_spending': rd_spending,
            'employment': employment,
            'profit_margin': profit_margin,
            'industry': firm_industry[i],
            'firm_quality': firm_quality[i]  # Include for analysis (normally unobserved)
        })

df = pd.DataFrame(data)

print(f"Panel data created:")
print(f"  Firms: {n_firms}")
print(f"  Years: {n_years} (2008-{2008 + n_years - 1})")
print(f"  Total observations: {len(df)}")
print(f"\nFirst 10 rows:")
print(df.head(10))

In [ ]:
section_divider("2. Basic Panel Summary Statistics")

## 2. Basic Panel Summary Statistics

Standard summary statistics don't account for panel structure. We need **within** and **between** statistics.

In [ ]:
# Standard summary statistics
print("Standard Summary Statistics:")
print("=" * 70)
print(df[['sales', 'rd_spending', 'employment', 'profit_margin']].describe())

### Panel-Specific Summary Statistics

Decompose variation into:
- **Overall:** Total variation across all observations
- **Between:** Variation between entity means
- **Within:** Variation around entity means (over time)

In [ ]:
def panel_summary_statistics(df, entity_col, var_cols):
    """
    Compute panel summary statistics: overall, between, and within.
    
    Parameters
    ----------
    df : pd.DataFrame
    entity_col : str
    var_cols : list
    
    Returns
    -------
    pd.DataFrame
        Summary statistics
    """
    results = []
    
    for var in var_cols:
        # Overall statistics
        overall_mean = df[var].mean()
        overall_std = df[var].std()
        overall_min = df[var].min()
        overall_max = df[var].max()
        
        # Between statistics (entity means)
        entity_means = df.groupby(entity_col)[var].mean()
        between_std = entity_means.std()
        between_min = entity_means.min()
        between_max = entity_means.max()
        
        # Within statistics (deviations from entity means)
        entity_means_expanded = df.groupby(entity_col)[var].transform('mean')
        within_deviations = df[var] - entity_means_expanded
        within_std = within_deviations.std()
        within_min = within_deviations.min()
        within_max = within_deviations.max()
        
        results.append({
            'Variable': var,
            'Overall Mean': overall_mean,
            'Overall Std': overall_std,
            'Between Std': between_std,
            'Within Std': within_std,
            'Min': overall_min,
            'Max': overall_max
        })
    
    return pd.DataFrame(results)

# Compute panel summary statistics
summary_stats = panel_summary_statistics(
    df, 
    'firm_id', 
    ['sales', 'rd_spending', 'employment', 'profit_margin']
)

print("\nPanel Summary Statistics:")
print("=" * 90)
print(summary_stats.to_string(index=False))

### Interpretation

**Between Std > Within Std:**
- Most variation is **between firms** (cross-sectional)
- Little variation **within firms** over time
- Example: Employment (firms differ in size, but each firm's employment is stable)

**Within Std > Between Std:**
- Most variation is **within firms** over time
- Firms are similar to each other, but change over time
- Example: Profit margin (volatile, fluctuates yearly)

**Similar Between and Within:**
- Both sources of variation are important
- Example: Sales (firms differ, and each firm's sales change over time)

In [ ]:
# Visualize variation decomposition
fig, ax = plt.subplots(figsize=(10, 6))

variables = summary_stats['Variable']
x = np.arange(len(variables))
width = 0.35

between = summary_stats['Between Std']
within = summary_stats['Within Std']

ax.bar(x - width/2, between, width, label='Between (cross-sectional)', alpha=0.8)
ax.bar(x + width/2, within, width, label='Within (time-series)', alpha=0.8)

ax.set_xlabel('Variable', fontsize=12)
ax.set_ylabel('Standard Deviation', fontsize=12)
ax.set_title('Variation Decomposition: Between vs Within', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(variables, rotation=45, ha='right')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("Interpretation:")
print("  - Employment: Mostly between-firm variation (firm size differences)")
print("  - Profit margin: More within-firm variation (volatile over time)")
print("  - Sales: Balanced between and within variation")

In [ ]:
section_divider("3. Time Series Visualization")

## 3. Time Series Visualization

Visualize how variables evolve over time for different entities.

In [ ]:
# Plot sales over time for sample firms
sample_firms = np.random.choice(df['firm_id'].unique(), 8, replace=False)

fig, ax = plt.subplots(figsize=(14, 7))

for firm in sample_firms:
    firm_data = df[df['firm_id'] == firm].sort_values('year')
    ax.plot(firm_data['year'], firm_data['sales'], marker='o', label=f'Firm {firm}', alpha=0.7)

# Add recession shading
ax.axvspan(2008, 2009, alpha=0.2, color='red', label='Financial Crisis')
ax.axvspan(2020, 2020, alpha=0.2, color='orange', label='COVID-19')

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Sales', fontsize=12)
ax.set_title('Firm Sales Over Time (Sample Firms)', fontsize=14)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Observations:")
print("  - Common shocks visible (financial crisis, COVID)")
print("  - Firms follow similar trends but at different levels")
print("  - Some firms more volatile than others")

### Aggregate Time Series (Averaged Across Entities)

In [ ]:
# Compute time averages
time_averages = df.groupby('year')[['sales', 'rd_spending', 'profit_margin']].mean()

fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Sales
axes[0].plot(time_averages.index, time_averages['sales'], marker='o', linewidth=2, color='blue')
axes[0].axvspan(2008, 2009, alpha=0.2, color='red')
axes[0].axvspan(2020, 2020, alpha=0.2, color='orange')
axes[0].set_ylabel('Average Sales', fontsize=11)
axes[0].set_title('Time Series of Panel Averages', fontsize=14)
axes[0].grid(True, alpha=0.3)

# R&D
axes[1].plot(time_averages.index, time_averages['rd_spending'], marker='o', linewidth=2, color='green')
axes[1].axvspan(2008, 2009, alpha=0.2, color='red')
axes[1].axvspan(2020, 2020, alpha=0.2, color='orange')
axes[1].set_ylabel('Average R&D', fontsize=11)
axes[1].grid(True, alpha=0.3)

# Profit margin
axes[2].plot(time_averages.index, time_averages['profit_margin'], marker='o', linewidth=2, color='purple')
axes[2].axvspan(2008, 2009, alpha=0.2, color='red', label='Crisis')
axes[2].axvspan(2020, 2020, alpha=0.2, color='orange', label='COVID')
axes[2].set_xlabel('Year', fontsize=11)
axes[2].set_ylabel('Average Profit Margin', fontsize=11)
axes[2].legend(loc='lower right')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
section_divider("4. Cross-Sectional Visualization")

## 4. Cross-Sectional Visualization

Examine differences between entities at specific time points.

In [ ]:
# Cross-sectional distribution in 2015
df_2015 = df[df['year'] == 2015]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Sales distribution
axes[0].hist(df_2015['sales'], bins=20, edgecolor='black', alpha=0.7)
axes[0].axvline(df_2015['sales'].mean(), color='red', linestyle='--', linewidth=2, label='Mean')
axes[0].set_xlabel('Sales', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].set_title('Cross-Sectional Distribution (2015)', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# R&D distribution
axes[1].hist(df_2015['rd_spending'], bins=20, edgecolor='black', alpha=0.7, color='green')
axes[1].axvline(df_2015['rd_spending'].mean(), color='red', linestyle='--', linewidth=2, label='Mean')
axes[1].set_xlabel('R&D Spending', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].set_title('R&D Distribution (2015)', fontsize=12)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

# Industry comparison
industry_means = df_2015.groupby('industry')['sales'].mean().sort_values()
axes[2].barh(industry_means.index, industry_means.values, alpha=0.7)
axes[2].set_xlabel('Average Sales', fontsize=11)
axes[2].set_title('Average Sales by Industry (2015)', fontsize=12)
axes[2].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

In [ ]:
section_divider("5. Within and Between Scatter Plots")

## 5. Within and Between Scatter Plots

Visualize relationships using within and between variation.

In [ ]:
# Compute entity means (between variation)
firm_means = df.groupby('firm_id')[['sales', 'rd_spending']].mean().reset_index()
firm_means.columns = ['firm_id', 'sales_mean', 'rd_mean']

# Compute deviations from means (within variation)
df_merged = df.merge(firm_means, on='firm_id')
df_merged['sales_within'] = df_merged['sales'] - df_merged['sales_mean']
df_merged['rd_within'] = df_merged['rd_spending'] - df_merged['rd_mean']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Overall relationship (pooled)
axes[0].scatter(df['rd_spending'], df['sales'], alpha=0.3, s=20)
z = np.polyfit(df['rd_spending'], df['sales'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['rd_spending'].min(), df['rd_spending'].max(), 100)
axes[0].plot(x_line, p(x_line), 'r-', linewidth=2, label=f'Slope: {z[0]:.2f}')
axes[0].set_xlabel('R&D Spending', fontsize=11)
axes[0].set_ylabel('Sales', fontsize=11)
axes[0].set_title('Overall Relationship (Pooled)', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Between relationship (entity means)
axes[1].scatter(firm_means['rd_mean'], firm_means['sales_mean'], alpha=0.6, s=50, color='green')
z_between = np.polyfit(firm_means['rd_mean'], firm_means['sales_mean'], 1)
p_between = np.poly1d(z_between)
x_line_between = np.linspace(firm_means['rd_mean'].min(), firm_means['rd_mean'].max(), 100)
axes[1].plot(x_line_between, p_between(x_line_between), 'r-', linewidth=2, label=f'Slope: {z_between[0]:.2f}')
axes[1].set_xlabel('R&D Spending (Firm Mean)', fontsize=11)
axes[1].set_ylabel('Sales (Firm Mean)', fontsize=11)
axes[1].set_title('Between Relationship (Cross-sectional)', fontsize=12)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Within relationship (deviations)
axes[2].scatter(df_merged['rd_within'], df_merged['sales_within'], alpha=0.3, s=20, color='purple')
z_within = np.polyfit(df_merged['rd_within'], df_merged['sales_within'], 1)
p_within = np.poly1d(z_within)
x_line_within = np.linspace(df_merged['rd_within'].min(), df_merged['rd_within'].max(), 100)
axes[2].plot(x_line_within, p_within(x_line_within), 'r-', linewidth=2, label=f'Slope: {z_within[0]:.2f}')
axes[2].set_xlabel('R&D Spending (Demeaned)', fontsize=11)
axes[2].set_ylabel('Sales (Demeaned)', fontsize=11)
axes[2].set_title('Within Relationship (Time-series)', fontsize=12)
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print(f"  - Pooled slope: {z[0]:.2f} (confounded by between and within)")
print(f"  - Between slope: {z_between[0]:.2f} (high-R&D firms have higher sales)")
print(f"  - Within slope: {z_within[0]:.2f} (when firm increases R&D, sales change)")
print("\n  Fixed effects estimates the WITHIN slope (causal if exogeneity holds)")

## 6. Outlier Detection

Identify unusual observations that might affect estimation.

In [ ]:
def detect_panel_outliers(df, entity_col, time_col, var_col, threshold=3):
    """
    Detect outliers using within-entity standardization.
    
    Parameters
    ----------
    df : pd.DataFrame
    entity_col : str
    time_col : str
    var_col : str
    threshold : float
        Number of standard deviations for outlier threshold
    
    Returns
    -------
    pd.DataFrame
        DataFrame with outlier flags
    """
    df_copy = df.copy()
    
    # Compute entity-specific z-scores
    entity_mean = df_copy.groupby(entity_col)[var_col].transform('mean')
    entity_std = df_copy.groupby(entity_col)[var_col].transform('std')
    
    df_copy['z_score'] = (df_copy[var_col] - entity_mean) / entity_std
    df_copy['is_outlier'] = np.abs(df_copy['z_score']) > threshold
    
    return df_copy

# Detect outliers in sales
df_with_outliers = detect_panel_outliers(df, 'firm_id', 'year', 'sales', threshold=2.5)

n_outliers = df_with_outliers['is_outlier'].sum()
print(f"Outliers detected: {n_outliers} ({n_outliers/len(df)*100:.1f}% of observations)")

# Show outlier examples
outlier_examples = df_with_outliers[df_with_outliers['is_outlier']].sort_values('z_score', ascending=False).head(10)
print("\nTop 10 outliers (by z-score):")
print(outlier_examples[['firm_id', 'year', 'sales', 'z_score']])

### Visualize Outliers

In [ ]:
# Plot firm with outliers
if n_outliers > 0:
    outlier_firms = df_with_outliers[df_with_outliers['is_outlier']]['firm_id'].unique()[:3]
    
    fig, axes = plt.subplots(len(outlier_firms), 1, figsize=(12, 4 * len(outlier_firms)))
    if len(outlier_firms) == 1:
        axes = [axes]
    
    for i, firm in enumerate(outlier_firms):
        firm_data = df_with_outliers[df_with_outliers['firm_id'] == firm].sort_values('year')
        
        # Plot normal and outlier points
        normal_data = firm_data[~firm_data['is_outlier']]
        outlier_data = firm_data[firm_data['is_outlier']]
        
        axes[i].plot(normal_data['year'], normal_data['sales'], marker='o', color='blue', label='Normal')
        axes[i].scatter(outlier_data['year'], outlier_data['sales'], color='red', s=100, marker='X', 
                       zorder=5, label='Outlier')
        
        axes[i].set_xlabel('Year', fontsize=11)
        axes[i].set_ylabel('Sales', fontsize=11)
        axes[i].set_title(f'Firm {firm}: Sales with Outliers Highlighted', fontsize=12)
        axes[i].legend()
        axes[i].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("No outliers detected with current threshold.")

---

## Exercises

### Exercise 1: Correlation Structure Analysis

**Task:** Compute within and between correlations for sales and R&D spending.

**Expected Output:** Three correlation coefficients (overall, between, within)

In [ ]:
# YOUR CODE HERE
# ---------------

def compute_panel_correlations(df, entity_col, var1, var2):
    """
    Compute overall, between, and within correlations.
    
    Parameters
    ----------
    df : pd.DataFrame
    entity_col : str
    var1, var2 : str
    
    Returns
    -------
    dict
    """
    # TODO: Implement correlation decomposition
    pass

corr_results = None  # Replace with your implementation

In [ ]:
# AUTO-GRADED TESTS
def test_exercise_1():
    assert corr_results is not None, "Don't forget to implement your solution!"
    assert 'overall' in corr_results, "Missing overall correlation"
    assert 'between' in corr_results, "Missing between correlation"
    assert 'within' in corr_results, "Missing within correlation"
    
    # Correlations should be between -1 and 1
    for key in ['overall', 'between', 'within']:
        assert -1 <= corr_results[key] <= 1, f"{key} correlation out of valid range"
    
    print("✅ Exercise 1 passed!")
    print("\nCorrelation Structure (Sales vs R&D):")
    print("=" * 50)
    for key, val in corr_results.items():
        print(f"{key.capitalize():.<20} {val:.4f}")
    
    print("\nInterpretation:")
    if corr_results['between'] > corr_results['within']:
        print("  - Stronger between correlation: High-R&D firms have higher sales")
        print("  - Weaker within correlation: Within-firm R&D changes weakly related to sales")
    else:
        print("  - Stronger within correlation: R&D increases boost sales within firms")

# Uncomment to test
# test_exercise_1()

### Exercise 2: Growth Rate Calculation

**Task:** Compute annual growth rates for sales within each firm.

In [ ]:
# YOUR CODE HERE
# ---------------

def compute_growth_rates(df, entity_col, time_col, var_col):
    """
    Compute percentage growth rates.
    
    Parameters
    ----------
    df : pd.DataFrame
    entity_col : str
    time_col : str
    var_col : str
    
    Returns
    -------
    pd.DataFrame
        Original data with growth rate column
    """
    # TODO: Implement growth rate calculation
    # Formula: (X_t - X_{t-1}) / X_{t-1}
    pass

df_with_growth = None  # Replace with your implementation

In [ ]:
# AUTO-GRADED TESTS
def test_exercise_2():
    assert df_with_growth is not None, "Don't forget to implement your solution!"
    assert 'sales_growth' in df_with_growth.columns, "Missing sales_growth column"
    
    # First observation per firm should be NaN
    first_obs = df_with_growth.groupby('firm_id').first()
    assert first_obs['sales_growth'].isna().all(), "First observation growth should be NaN"
    
    # Check growth rate calculation for one firm
    firm1 = df_with_growth[df_with_growth['firm_id'] == 1].sort_values('year').reset_index(drop=True)
    expected_growth = (firm1.loc[1, 'sales'] - firm1.loc[0, 'sales']) / firm1.loc[0, 'sales']
    assert np.abs(firm1.loc[1, 'sales_growth'] - expected_growth) < 0.01, "Growth rate calculation incorrect"
    
    print("✅ Exercise 2 passed!")
    print("\nGrowth rate statistics:")
    print(df_with_growth['sales_growth'].describe())
    
    # Plot growth distribution
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(df_with_growth['sales_growth'].dropna(), bins=50, edgecolor='black', alpha=0.7)
    ax.axvline(0, color='red', linestyle='--', linewidth=2)
    ax.set_xlabel('Sales Growth Rate', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title('Distribution of Annual Sales Growth Rates', fontsize=12)
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()

# Uncomment to test
# test_exercise_2()

### Exercise 3: Industry-Specific Time Trends

**Task:** Visualize average sales trends by industry.

In [ ]:
# YOUR CODE HERE
# ---------------

def plot_industry_trends(df, time_col, var_col, industry_col):
    """
    Plot variable trends by industry.
    
    Parameters
    ----------
    df : pd.DataFrame
    time_col : str
    var_col : str
    industry_col : str
    """
    # TODO: Create industry trend plot
    pass

# Uncomment to run
# plot_industry_trends(df, 'year', 'sales', 'industry')

In [ ]:
# Verification (no auto-grading, visual check)
print("Exercise 3: Visual verification")
print("Your plot should show:")
print("  - Separate line for each industry")
print("  - Time on x-axis, average sales on y-axis")
print("  - Legend identifying industries")
print("  - Grid for readability")

---

## Summary

### Key Takeaways

1. **Within vs Between Variation:** Understanding variation structure guides model selection

2. **Panel Summary Statistics:** Standard statistics insufficient; compute within/between separately

3. **Visual Exploration:** Both time series and cross-sectional plots essential

4. **Relationship Decomposition:** Overall correlation differs from within and between

5. **Outlier Detection:** Entity-specific standardization better than overall

### Model Selection Guidance

**Use Fixed Effects when:**
- Sufficient within variation in X
- Between and within relationships differ substantially
- Concerned about time-invariant confounders

**Consider Random Effects when:**
- Little within variation
- Between relationship is of interest
- Time-invariant variables are important

**Pooled OLS only when:**
- No entity heterogeneity visible
- Between and within relationships similar
- Theory suggests no confounding

### What's Next

In Module 2:
- Implement fixed effects models
- Estimate causal within-entity effects
- Test FE assumptions
- Handle time effects

---

**Next:** Module 2 - Fixed Effects Estimation

In [ ]:
key_takeaways(["Module 1.1 (Data Preparation) completed", "Understanding of summary statistics", "Basic visualization skills"])